# MGMT298D: Science and Strategy of AI
### Week 7B - Transformers
### Application: Text Classification and Translation

## Import Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import pad_sequences
import re
import string

np.random.seed(42)
tf.random.set_seed(42)

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBED_DIM = 128
NUM_CLASSES = 46

# Load Reuters dataset (46 news categories)
(x_train, y_train), (x_test, y_test) = keras.datasets.reuters.load_data(num_words=VOCAB_SIZE)
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print(f"Training: {x_train.shape} | Test: {x_test.shape}")

## Build Transformer Components

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

print("Transformer components defined")

## Model A: Simple Transformer Classifier

In [ ]:
inputs = layers.Input(shape=(MAX_LEN,))
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
x = TransformerBlock(EMBED_DIM, num_heads=2, ff_dim=32)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_A = keras.Model(inputs=inputs, outputs=outputs)
model_A.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_A = model_A.fit(
    x_train, y_train, batch_size=128, epochs=20, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)],
    verbose=1
)

loss_A, acc_A = model_A.evaluate(x_test, y_test)
print(f"Model A — Test Accuracy: {acc_A*100:.2f}%")

## Model B: Deeper Transformer (3 Blocks)

In [ ]:
inputs = layers.Input(shape=(MAX_LEN,))
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)

# Stack 3 transformer blocks
for _ in range(3):
    x = TransformerBlock(EMBED_DIM, num_heads=4, ff_dim=64)(x)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_B = keras.Model(inputs=inputs, outputs=outputs)
model_B.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_B = model_B.fit(
    x_train, y_train, batch_size=128, epochs=40, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)],
    verbose=1
)

loss_B, acc_B = model_B.evaluate(x_test, y_test)
print(f"Model B — Test Accuracy: {acc_B*100:.2f}%")

## Text Generation with GPT-2

In [ ]:
!pip install transformers -q

from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = TFGPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)
print("GPT-2 model loaded")

In [ ]:
# Generation parameters - experiment with these!
prompt = "In a shocking finding, scientists discovered"
max_length = 100
temperature = 1.0   # Higher = more creative, lower = more focused
top_k = 50          # Consider only top-k most likely words
top_p = 0.95        # Nucleus sampling threshold

input_ids = tokenizer.encode(prompt, return_tensors='tf')

output = gpt2_model.generate(
    input_ids, do_sample=True, max_length=max_length,
    temperature=temperature, top_k=top_k, top_p=top_p
)

print(f"Prompt: {prompt}\n")
print("Generated text:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['A (1 Block, 2 Heads)', 'B (3 Blocks, 4 Heads)'],
    'Test Accuracy': [acc_A * 100, acc_B * 100],
    'Parameters': [model_A.count_params(), model_B.count_params()]
})
results = results.set_index('Model')
print(results.to_string(formatters={'Test Accuracy': '{:.2f}%'.format, 'Parameters': '{:,}'.format}))

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_A.history['val_accuracy'], label='Model A')
ax1.plot(history_B.history['val_accuracy'], label='Model B')
ax1.set_title('Validation Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)

ax2.plot(history_A.history['val_loss'], label='Model A')
ax2.plot(history_B.history['val_loss'], label='Model B')
ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.show()